# 🎸 Guitar Music Transcription — Transformer-based AMT Pipeline

**Author:** Reneth Saju · Columbia University, MS Computer Science  
**Dataset:** [GuitarSet](https://guitarset.weebly.com/) — 360 guitar recordings with JAMS annotations  
**Keywords:** Automatic Music Transcription · Transformer · Audio ML · GuitarSet · PyTorch

---

> **Abstract:** This project implements a Transformer-based automatic music transcription (AMT) pipeline trained on the GuitarSet dataset. The model jointly predicts frame-level onset detection, pitch classification (128 MIDI classes + silence), and velocity estimation from log-mel spectrogram features of individual guitar strings.


## 📋 Table of Contents

1. [Environment Check](#1-environment-check)
2. [Package Installation](#2-package-installation)
3. [Configuration & Core Definitions](#3-configuration--core-definitions)
4. [Dataset Loading — GuitarSet](#4-dataset-loading--guitarset)
5. [GuitarSet Parsing](#5-guitarset-parsing)
6. [Data Processing & Splits](#6-data-processing--splits)
7. [Model Initialisation](#7-model-initialisation)
8. [Training](#8-training)
9. [Visualise Training Curves](#9-visualise-training-curves)
10. [Inference & Evaluation](#10-inference--evaluation)


---
## 1. Environment Check

Verify Python version and Colab environment before installation.


In [ ]:
import sys
print("=" * 60)
print("INITIAL ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python version: {sys.version}")
print(f"Running in Colab: {('google.colab' in str(get_ipython()))}")
print("=" * 60)

---
## 2. Package Installation

Installs all required dependencies and tests three MIDI synthesis backends (pyfluidsynth → midi2audio → timidity), selecting the best available. Takes ~3–5 minutes on first run.

> **Runtime note:** This notebook is designed for **Google Colab** with GPU acceleration. Enable GPU via `Runtime → Change runtime type → T4 GPU`.


In [ ]:
"""
CELL 2: Complete Package Installation + FluidSynth Fix (ALL-IN-ONE)

Installs everything and fixes FluidSynth automatically.
Run this ONE cell and you're ready to go!
"""
!pip install -q jams
!pip install -q jams pretty_midi librosa soundfile
import subprocess
import sys
import os

print("🔧 Complete Package Installation")
print("=" * 60)
print("This installs everything and fixes FluidSynth")
print("Takes ~3-5 minutes")
print("=" * 60)
print()

# ============================================
# STEP 1: System Packages
# ============================================
print("Step 1/5: Installing system packages...")
!apt-get update -qq 2>&1 | grep -v "Skipping acquire"
!apt-get install -y -qq fluidsynth timidity
print("✅ FluidSynth and Timidity installed")

# ============================================
# STEP 2: Python Audio Packages
# ============================================
print("\nStep 2/5: Installing Python audio packages...")
!pip install -q librosa soundfile pretty_midi
print("✅ Audio packages installed")

# ============================================
# STEP 3: FluidSynth Python Wrappers (Multiple Methods)
# ============================================
print("\nStep 3/5: Installing MIDI synthesis libraries...")
!pip uninstall -y pyfluidsynth -qq
!pip install -q pyfluidsynth==1.3.0
!pip install -q midi2audio
print("✅ MIDI synthesis libraries installed")

# ============================================
# STEP 4: ML & Visualization Packages
# ============================================
print("\nStep 4/5: Installing ML packages...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q matplotlib seaborn scikit-learn tqdm jams
print("✅ ML packages installed")

# ============================================
# STEP 5: Test & Choose Best Synthesis Method
# ============================================
print("\nStep 5/5: Testing MIDI synthesis methods...")
print("-" * 60)

import pretty_midi
import numpy as np
import soundfile as sf

# Create test MIDI
test_midi = pretty_midi.PrettyMIDI()
instrument = pretty_midi.Instrument(0)
note = pretty_midi.Note(velocity=80, pitch=60, start=0, end=0.5)
instrument.notes.append(note)
test_midi.instruments.append(instrument)

# Test Method 1: pyfluidsynth
method_1_works = False
try:
    audio1 = test_midi.fluidsynth(fs=22050)
    print("✅ Method 1 (pyfluidsynth): Working!")
    method_1_works = True
except Exception as e:
    print(f"❌ Method 1 (pyfluidsynth): {str(e)[:50]}")

# Test Method 2: midi2audio
method_2_works = False
try:
    from midi2audio import FluidSynth
    test_midi.write('/tmp/test.mid')
    fs_synth = FluidSynth()
    fs_synth.midi_to_audio('/tmp/test.mid', '/tmp/test.wav')
    audio2, sr = sf.read('/tmp/test.wav')
    print("✅ Method 2 (midi2audio): Working!")
    method_2_works = True
except Exception as e:
    print(f"❌ Method 2 (midi2audio): {str(e)[:50]}")

# Test Method 3: timidity
method_3_works = False
try:
    test_midi.write('/tmp/test.mid')
    result = subprocess.run(['timidity', '/tmp/test.mid', '-Ow', '-o', '/tmp/test_timidity.wav'],
                          capture_output=True, timeout=5)
    if result.returncode == 0:
        audio3, sr = sf.read('/tmp/test_timidity.wav')
        print("✅ Method 3 (timidity): Working!")
        method_3_works = True
    else:
        raise Exception("timidity command failed")
except Exception as e:
    print(f"❌ Method 3 (timidity): {str(e)[:50]}")

# Choose best method
print("\n" + "=" * 60)
if method_1_works:
    print("🎵 Selected: Method 1 (pyfluidsynth) - Best quality")
    SYNTH_METHOD = 1
elif method_2_works:
    print("🎵 Selected: Method 2 (midi2audio) - Good quality")
    SYNTH_METHOD = 2
elif method_3_works:
    print("🎵 Selected: Method 3 (timidity) - Reliable fallback")
    SYNTH_METHOD = 3
else:
    print("🎵 Selected: Method 0 (pure tones) - Basic fallback")
    SYNTH_METHOD = 0
print("=" * 60)

# ============================================
# VERIFICATION
# ============================================
print("\n" + "=" * 60)
print("VERIFYING ALL PACKAGES")
print("=" * 60)

try:
    import numpy as np
    print(f"✅ NumPy {np.__version__}")

    import pandas as pd
    print(f"✅ Pandas {pd.__version__}")

    import librosa
    print(f"✅ Librosa {librosa.__version__}")

    import soundfile
    print(f"✅ SoundFile")

    import pretty_midi
    print(f"✅ Pretty MIDI")

    import torch
    print(f"✅ PyTorch {torch.__version__}")

    import matplotlib.pyplot as plt
    print(f"✅ Matplotlib")

    import seaborn
    print(f"✅ Seaborn")

    import jams
    print(f"✅ JAMS")

    print(f"✅ MIDI Synthesis (Method {SYNTH_METHOD})")

    # Check GPU
    if torch.cuda.is_available():
        print(f"\n🎮 GPU DETECTED:")
        print(f"   {torch.cuda.get_device_name(0)}")
        print(f"   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")
    else:
        print(f"\n⚠️  No GPU detected - using CPU")

    print("\n" + "=" * 60)
    print("✅ ALL PACKAGES INSTALLED AND WORKING!")
    print(f"   MIDI Synthesis Method: {SYNTH_METHOD}")
    print("=" * 60)

    # Clean up test files
    for f in ['/tmp/test.mid', '/tmp/test.wav', '/tmp/test_timidity.wav']:
        if os.path.exists(f):
            os.remove(f)

except Exception as e:
    print(f"\n❌ Verification error: {e}")

print("\n💡 Ready to generate synthetic data!")
print("   Run Cell S1 next to create 500 samples")


---
## 3. Configuration & Core Definitions

Defines all core components in one cell:
- **`TranscriptionConfig`** — central hyperparameter dataclass
- **`AudioFeatureExtractor`** — raw audio → log-mel spectrogram `(T, 128)`
- **`MidiLabelParser`** — MIDI file → frame-level onset/pitch/velocity tensors
- **`MusicDataset`** — PyTorch Dataset pairing audio features with MIDI labels
- **`MusicTranscriptionModel`** — 3-layer Transformer encoder with three output heads
- **`MultiTaskLoss`** — weighted BCE (onset) + CrossEntropy (pitch) + MSE (velocity)
- **`train_epoch` / `validate_epoch`** — training loop utilities

| Hyperparameter | Value |
|----------------|-------|
| Sample rate | 22,050 Hz |
| Mel bins | 128 |
| Max frames | 512 |
| Model dim (`D_MODEL`) | 128 |
| Transformer layers | 3 |
| Attention heads | 4 |
| Learning rate | 3e-4 |
| Batch size | 16 |
| Epochs | 30 |


In [ ]:
"""
CELL 3: Configuration & Core Definitions

Defines the central config object, model architecture, dataset class,
feature extractor, loss function, and training utilities.
All subsequent cells depend on this — run before anything else.
"""

import os
import torch
import torch.nn as nn
import numpy as np
import librosa
import pretty_midi
from torch.utils.data import Dataset
from dataclasses import dataclass
from typing import Optional


# ──────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────

@dataclass
class TranscriptionConfig:
    # Audio
    SAMPLE_RATE: int = 22050
    N_FFT: int = 2048
    HOP_LENGTH: int = 512
    N_MELS: int = 128
    MAX_FRAMES: int = 512          # frames per sample

    # MIDI
    NUM_PITCH_CLASSES: int = 129   # 128 MIDI pitches + silence token (128)
    FPS: float = SAMPLE_RATE / HOP_LENGTH  # ~43 fps

    # Model
    D_MODEL: int = 128             # transformer embedding dim
    N_HEADS: int = 4
    N_LAYERS: int = 3
    D_FF: int = 256
    DROPOUT: float = 0.1

    # Training
    BATCH_SIZE: int = 16
    NUM_EPOCHS: int = 30
    LEARNING_RATE: float = 3e-4
    WEIGHT_DECAY: float = 1e-5

    # Paths
    CHECKPOINT_DIR: str = '/content/checkpoints'
    DATA_DIR: str = '/content/music_data'

config = TranscriptionConfig()
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(config.DATA_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Config loaded | Device: {device}")
print(f"   Sample rate: {config.SAMPLE_RATE} Hz | Hop: {config.HOP_LENGTH} | FPS: {config.FPS:.1f}")
print(f"   Max frames: {config.MAX_FRAMES} | Mel bins: {config.N_MELS}")


# ──────────────────────────────────────────────
# Audio Feature Extractor
# ──────────────────────────────────────────────

class AudioFeatureExtractor:
    """Convert raw audio to log-mel spectrogram frames."""

    def __init__(self, cfg: TranscriptionConfig):
        self.cfg = cfg

    def extract(self, audio_path: str) -> torch.Tensor:
        """
        Returns: FloatTensor of shape (MAX_FRAMES, N_MELS)
        """
        audio, _ = librosa.load(audio_path, sr=self.cfg.SAMPLE_RATE, mono=True)

        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=self.cfg.SAMPLE_RATE,
            n_fft=self.cfg.N_FFT,
            hop_length=self.cfg.HOP_LENGTH,
            n_mels=self.cfg.N_MELS,
        )
        log_mel = librosa.power_to_db(mel, ref=np.max).T  # (T, N_MELS)

        # Pad or crop to MAX_FRAMES
        T = log_mel.shape[0]
        if T >= self.cfg.MAX_FRAMES:
            log_mel = log_mel[:self.cfg.MAX_FRAMES]
        else:
            pad = np.zeros((self.cfg.MAX_FRAMES - T, self.cfg.N_MELS), dtype=np.float32)
            log_mel = np.vstack([log_mel, pad])

        return torch.tensor(log_mel, dtype=torch.float32)


# ──────────────────────────────────────────────
# MIDI Label Parser
# ──────────────────────────────────────────────

class MidiLabelParser:
    """Convert a MIDI file into frame-level onset / pitch / velocity tensors."""

    def __init__(self, cfg: TranscriptionConfig):
        self.cfg = cfg

    def parse(self, midi_path: str):
        """
        Returns:
          onset    : (MAX_FRAMES,)  float32  — 1 at onset frames, else 0
          pitch    : (MAX_FRAMES,)  long     — MIDI pitch (0-127) or 128 (silence)
          velocity : (MAX_FRAMES,)  float32  — normalised velocity [0, 1]
        """
        onset    = torch.zeros(self.cfg.MAX_FRAMES, dtype=torch.float32)
        pitch    = torch.full((self.cfg.MAX_FRAMES,), 128, dtype=torch.long)  # silence
        velocity = torch.zeros(self.cfg.MAX_FRAMES, dtype=torch.float32)

        try:
            midi = pretty_midi.PrettyMIDI(midi_path)
        except Exception:
            return onset, pitch, velocity

        for instrument in midi.instruments:
            for note in instrument.notes:
                onset_frame = int(note.start * self.cfg.FPS)
                end_frame   = int(note.end   * self.cfg.FPS)

                if onset_frame >= self.cfg.MAX_FRAMES:
                    continue

                onset_frame = min(onset_frame, self.cfg.MAX_FRAMES - 1)
                end_frame   = min(end_frame,   self.cfg.MAX_FRAMES)

                onset[onset_frame] = 1.0
                pitch[onset_frame:end_frame] = note.pitch
                velocity[onset_frame:end_frame] = note.velocity / 127.0

        return onset, pitch, velocity


# ──────────────────────────────────────────────
# Dataset
# ──────────────────────────────────────────────

extractor = AudioFeatureExtractor(config)
parser    = MidiLabelParser(config)

class MusicDataset(Dataset):
    """Pairs (log-mel spectrogram) with (onset, pitch, velocity) labels."""

    def __init__(self, audio_paths, midi_paths, feature_extractor, label_parser):
        self.audio_paths = audio_paths
        self.midi_paths  = midi_paths
        self.extractor   = feature_extractor
        self.parser      = label_parser

    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, idx):
        features           = self.extractor.extract(self.audio_paths[idx])
        onset, pitch, vel  = self.parser.parse(self.midi_paths[idx])
        return features, onset, pitch, vel


def collate_fn(batch):
    features  = torch.stack([b[0] for b in batch])   # (B, T, N_MELS)
    onset     = torch.stack([b[1] for b in batch])   # (B, T)
    pitch     = torch.stack([b[2] for b in batch])   # (B, T)
    velocity  = torch.stack([b[3] for b in batch])   # (B, T)
    return features, onset, pitch, velocity


# ──────────────────────────────────────────────
# Model
# ──────────────────────────────────────────────

class MusicTranscriptionModel(nn.Module):
    """
    Transformer-based automatic music transcription model.

    Input  : log-mel spectrogram  (B, T, N_MELS)
    Outputs:
      onset    : (B, T, 1)         — onset probability logit
      pitch    : (B, T, 129)       — pitch class logits (128 MIDI + silence)
      velocity : (B, T, 1)         — velocity [0, 1]
    """

    def __init__(self, cfg: TranscriptionConfig):
        super().__init__()
        self.cfg = cfg

        # Input projection: N_MELS → D_MODEL
        self.input_proj = nn.Linear(cfg.N_MELS, cfg.D_MODEL)

        # Positional encoding
        self.pos_enc = nn.Embedding(cfg.MAX_FRAMES, cfg.D_MODEL)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.D_MODEL,
            nhead=cfg.N_HEADS,
            dim_feedforward=cfg.D_FF,
            dropout=cfg.DROPOUT,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.N_LAYERS)

        # Output heads
        self.onset_head    = nn.Linear(cfg.D_MODEL, 1)
        self.pitch_head    = nn.Linear(cfg.D_MODEL, cfg.NUM_PITCH_CLASSES)
        self.velocity_head = nn.Sequential(nn.Linear(cfg.D_MODEL, 1), nn.Sigmoid())

        self.dropout = nn.Dropout(cfg.DROPOUT)

    def forward(self, x):
        # x: (B, T, N_MELS)
        B, T, _ = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)

        h = self.input_proj(x) + self.pos_enc(positions)   # (B, T, D_MODEL)
        h = self.dropout(h)
        h = self.transformer(h)                             # (B, T, D_MODEL)

        return {
            'onset':    self.onset_head(h).squeeze(-1),    # (B, T)
            'pitch':    self.pitch_head(h),                # (B, T, 129)
            'velocity': self.velocity_head(h).squeeze(-1) # (B, T)
        }


# ──────────────────────────────────────────────
# Loss
# ──────────────────────────────────────────────

class MultiTaskLoss(nn.Module):
    """Weighted sum of onset BCE + pitch CrossEntropy + velocity MSE."""

    def __init__(self, w_onset=1.0, w_pitch=10.0, w_velocity=1.0):
        super().__init__()
        self.w_onset    = w_onset
        self.w_pitch    = w_pitch
        self.w_velocity = w_velocity
        self.onset_loss    = nn.BCEWithLogitsLoss()
        self.pitch_loss    = nn.CrossEntropyLoss()
        self.velocity_loss = nn.MSELoss()

    def forward(self, predictions, onset_targets, pitch_targets, velocity_targets):
        l_onset = self.onset_loss(predictions['onset'], onset_targets)
        # CrossEntropy expects (B, C, T)
        l_pitch = self.pitch_loss(
            predictions['pitch'].permute(0, 2, 1),
            pitch_targets
        )
        l_vel = self.velocity_loss(predictions['velocity'], velocity_targets)

        total = self.w_onset * l_onset + self.w_pitch * l_pitch + self.w_velocity * l_vel
        return {
            'total':    total,
            'onset':    l_onset.item(),
            'pitch':    l_pitch.item(),
            'velocity': l_vel.item(),
        }


# ──────────────────────────────────────────────
# Training utilities
# ──────────────────────────────────────────────

def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    totals = {'total': 0.0, 'onset': 0.0, 'pitch': 0.0, 'velocity': 0.0}

    for features, onset, pitch, velocity in dataloader:
        features = features.to(device)
        onset    = onset.to(device)
        pitch    = pitch.to(device)
        velocity = velocity.to(device)

        optimizer.zero_grad()
        preds  = model(features)
        losses = criterion(preds, onset, pitch, velocity)
        losses['total'].backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        for k in totals:
            totals[k] += losses[k] if k == 'total' else losses[k]

    n = len(dataloader)
    return {k: v / n for k, v in totals.items()}


def validate_epoch(model, dataloader, criterion):
    model.eval()
    totals = {'total': 0.0, 'onset': 0.0, 'pitch': 0.0, 'velocity': 0.0}

    with torch.no_grad():
        for features, onset, pitch, velocity in dataloader:
            features = features.to(device)
            onset    = onset.to(device)
            pitch    = pitch.to(device)
            velocity = velocity.to(device)

            preds  = model(features)
            losses = criterion(preds, onset, pitch, velocity)

            for k in totals:
                totals[k] += losses[k] if k == 'total' else losses[k]

    n = len(dataloader)
    return {k: v / n for k, v in totals.items()}


print("\n✅ All core definitions loaded:")
print("   TranscriptionConfig, AudioFeatureExtractor, MidiLabelParser")
print("   MusicDataset, collate_fn")
print("   MusicTranscriptionModel, MultiTaskLoss")
print("   train_epoch, validate_epoch")


---
## 4. Dataset Loading — GuitarSet

Mounts Google Drive and copies the GuitarSet dataset to local Colab storage (`/content/guitarset`) for faster I/O during training.

**GuitarSet** contains 360 recordings of guitar performances across 6 players, 5 musical styles, and multiple keys — each annotated with per-string JAMS files providing note-level ground truth.

> **Setup:** Upload your GuitarSet folder to Google Drive at `MyDrive/Guitarset Dataset/` before running this cell.


In [ ]:
"""
CELL G1: Mount Drive and Locate GuitarSet Files
"""

from google.colab import drive
from pathlib import Path
import shutil, os, time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Mount Drive ───────────────────────────────────────────────────────────────
print("📂 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

drive_dataset_path = Path('/content/drive/MyDrive/Guitarset Dataset')
local_guitarset_path = Path('/content/guitarset')
local_audio_dir      = local_guitarset_path / 'audio'
local_annotation_dir = local_guitarset_path / 'annotation'

local_audio_dir.mkdir(parents=True, exist_ok=True)
local_annotation_dir.mkdir(parents=True, exist_ok=True)

print(f"\n🔍 Looking for GuitarSet in: {drive_dataset_path}")

if not drive_dataset_path.exists():
    raise FileNotFoundError(f"Dataset folder not found: {drive_dataset_path}")

# ── Scan files ────────────────────────────────────────────────────────────────
audio_files_in_drive = list(drive_dataset_path.glob('**/*.wav'))
jams_files_in_drive  = list(drive_dataset_path.glob('**/*.jams'))

print(f"\n📊 Found in Drive:")
print(f"   Audio files (.wav): {len(audio_files_in_drive)}")
print(f"   Annotation files (.jams): {len(jams_files_in_drive)}")

# Check how many are already local
already_audio = len(list(local_audio_dir.glob('*.wav')))
already_jams  = len(list(local_annotation_dir.glob('*.jams')))
print(f"\n📁 Already in local storage:")
print(f"   Audio: {already_audio}/360 | Annotations: {already_jams}/360")

if already_audio == 360 and already_jams == 360:
    print("\n✅ All files already local — skipping copy.")
else:
    # ── Parallel copy with retry ──────────────────────────────────────────────
    def copy_with_retry(src, dst, max_retries=5, wait=3):
        for attempt in range(1, max_retries + 1):
            try:
                shutil.copy2(src, dst)
                return True
            except OSError as e:
                if attempt == max_retries:
                    return False
                time.sleep(wait * attempt)
        return False

    def copy_file(args):
        src, dst = args
        if dst.exists():
            return True
        return copy_with_retry(src, dst)

    # Build job lists (only files not yet copied)
    audio_jobs = [(f, local_audio_dir / f.name)
                  for f in audio_files_in_drive
                  if not (local_audio_dir / f.name).exists()]
    jams_jobs  = [(f, local_annotation_dir / f.name)
                  for f in jams_files_in_drive
                  if not (local_annotation_dir / f.name).exists()]

    all_jobs = audio_jobs + jams_jobs
    print(f"\n📦 Copying {len(all_jobs)} files using parallel threads...")

    failed = []
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {executor.submit(copy_file, job): job for job in all_jobs}
        with tqdm(total=len(all_jobs), desc="Copying") as pbar:
            for future in as_completed(futures):
                if not future.result():
                    failed.append(futures[future][0].name)
                pbar.update(1)

    if failed:
        print(f"\n⚠️  {len(failed)} files failed: {failed[:5]}")

    # ── Summary ───────────────────────────────────────────────────────────────
    local_audio_count = len(list(local_audio_dir.glob('*.wav')))
    local_jams_count  = len(list(local_annotation_dir.glob('*.jams')))
    print(f"\n✅ Done!")
    print(f"   Audio:       {local_audio_count}/360")
    print(f"   Annotations: {local_jams_count}/360")

if len(list(local_audio_dir.glob('*.wav'))) > 0:
    print("\n✅ GuitarSet ready for training!")
else:
    print("\n⚠️  Something went wrong — check errors above.")

---
## 5. GuitarSet Parsing

`GuitarSetParser` handles the non-trivial filename matching between audio files (e.g. `00_BN1-129-Eb_comp_hex_cln.wav`) and their JAMS annotations (e.g. `00_BN1-129-Eb_comp.jams`). All 360 audio files are matched 1-to-1 with annotations.


In [ ]:
"""
CELL G2 (FINAL FIX): Parse GuitarSet with Correct Filename Matching
"""

import jams
import librosa
import soundfile as sf
import pretty_midi
from pathlib import Path
from tqdm import tqdm
import numpy as np

class GuitarSetParser:
    """Parse GuitarSet dataset from local storage"""

    def __init__(self, guitarset_root='/content/guitarset'):
        self.root = Path(guitarset_root)
        self.audio_dir = self.root / 'audio'
        self.annotation_dir = self.root / 'annotation'

        print(f"📁 GuitarSet root: {self.root}")
        print(f"📁 Audio directory: {self.audio_dir}")
        print(f"📁 Annotation directory: {self.annotation_dir}")

    def get_audio_files(self):
        """Get all audio files"""
        files = sorted(list(self.audio_dir.glob('*.wav')))
        print(f"   Found {len(files)} audio files")
        return files

    def extract_string_audio(self, audio_path, string_idx=0):
        """Extract one string from audio"""

        try:
            audio, sr = librosa.load(audio_path, sr=22050, mono=False)
        except:
            audio, sr = librosa.load(audio_path, sr=22050, mono=True)
            return audio

        if audio.ndim == 1:
            return audio

        # Multi-channel - extract specific string
        if string_idx < audio.shape[0]:
            return audio[string_idx]
        else:
            return audio[0]

    def get_jams_path(self, audio_path):
        """
        Get corresponding JAMS annotation file

        GuitarSet naming:
        - Audio: 00_BN1-129-Eb_solo_hex_cln.wav
        - Annotation: 00_BN1-129-Eb_solo.jams

        Keep solo/comp, remove hex/cln suffixes
        """

        stem = audio_path.stem

        # Strategy: Remove only the technical suffixes, keep solo/comp
        # Remove in specific order
        suffixes_to_remove = [
            '_hex_cln',
            '_hex',
            '_cln',
        ]

        for suffix in suffixes_to_remove:
            stem = stem.replace(suffix, '')

        # Now stem should be like: 00_BN1-129-Eb_solo or 00_BN1-129-Eb_comp
        jams_path = self.annotation_dir / f'{stem}.jams'

        if jams_path.exists():
            return jams_path

        return None

    def parse_jams(self, jams_path):
        """Parse JAMS file and extract notes"""

        if not jams_path or not jams_path.exists():
            return [], 0

        try:
            jam = jams.load(str(jams_path))
        except Exception as e:
            return [], 0

        # Collect all notes
        all_notes = []

        for ann in jam.annotations:
            if ann.namespace == 'note_midi':
                for note_data in ann.data:
                    all_notes.append({
                        'pitch': int(note_data.value),
                        'start': float(note_data.time),
                        'end': float(note_data.time + note_data.duration),
                        'velocity': 80
                    })

        # Get duration
        try:
            duration = float(jam.file_metadata.duration)
        except:
            duration = 30.0

        return all_notes, duration

# Create parser
print("🎸 Initializing GuitarSet parser...")
gs_parser = GuitarSetParser()

# Get audio files
audio_files = gs_parser.get_audio_files()

if len(audio_files) > 0:
    print(f"\n✅ GuitarSet parser ready!")
    print(f"   Audio files: {len(audio_files)}")

    # Test annotation matching
    print(f"\n🔍 Testing annotation matching...")

    matched = 0
    unmatched = 0

    for i, test_file in enumerate(audio_files[:10]):
        test_jams = gs_parser.get_jams_path(test_file)

        if test_jams and test_jams.exists():
            matched += 1
            if i < 3:
                print(f"   ✅ {test_file.name}")
                print(f"      → {test_jams.name}")
        else:
            unmatched += 1
            if i < 3:
                print(f"   ❌ {test_file.name}")

                # Show what we tried
                stem = test_file.stem
                for suffix in ['_hex_cln', '_hex', '_cln']:
                    stem = stem.replace(suffix, '')
                print(f"      → Tried: {stem}.jams (not found)")

    print(f"\n📊 Annotation matching (first 10 files):")
    print(f"   Matched: {matched}/10")
    print(f"   Unmatched: {unmatched}/10")

    # Count total matches
    print(f"\n📊 Checking all {len(audio_files)} files...")
    total_matched = sum(1 for f in audio_files if gs_parser.get_jams_path(f) is not None)
    print(f"   Total matched: {total_matched}/{len(audio_files)}")

    if total_matched > 0:
        # Test parse
        for test_file in audio_files:
            test_jams = gs_parser.get_jams_path(test_file)
            if test_jams:
                test_notes, test_duration = gs_parser.parse_jams(test_jams)
                print(f"\n✅ Sample annotation parsed:")
                print(f"   File: {test_file.name}")
                print(f"   Annotation: {test_jams.name}")
                print(f"   Notes: {len(test_notes)}")
                print(f"   Duration: {test_duration:.1f}s")
                break

    if total_matched == 0:
        print(f"\n⚠️ No annotations matched!")
        print(f"\n   Checking annotation directory structure...")
        ann_files = sorted(list(gs_parser.annotation_dir.glob('*.jams')))
        print(f"   Found {len(ann_files)} JAMS files")

        if len(ann_files) > 0:
            print(f"\n   Sample annotation names:")
            for ann in ann_files[:5]:
                print(f"   - {ann.name}")

            print(f"\n   Sample audio names:")
            for aud in audio_files[:5]:
                print(f"   - {aud.name}")
else:
    print(f"\n❌ No audio files found!")


---
## 6. Data Processing & Splits

Processes all 360 GuitarSet recordings by extracting each of the 6 individual guitar strings, producing **2,160 monophonic samples** (~970 minutes of audio). Each sample is saved as a WAV + MIDI pair.

The dataset is then split **80 / 10 / 10** (train / val / test) and wrapped in PyTorch `DataLoader`s.

| Split | Samples | Batches |
|-------|---------|--------|
| Train | 1,728 | 108 |
| Val | 216 | 14 |
| Test | 216 | 14 |

> ⏱️ Processing all 360 files takes ~10–15 minutes. This only needs to run once — processed files are cached to disk.


In [ ]:
"""
CELL G3 (FULL PROCESSING): Process ALL 360 GuitarSet Files

Creates ~2,160 samples (360 files × 6 strings)
Takes ~10-15 minutes but gives best results!
"""

import os
import shutil
import soundfile as sf
import pretty_midi
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

processed_local = Path('/content/music_data/guitarset_processed')
processed_drive = Path('/content/drive/MyDrive/guitarset_processed')

# ── Restore from Drive if already processed ───────────────────────────────────
if processed_drive.exists() and len(list(processed_drive.glob('*.wav'))) > 0:
    print("✅ Found processed data on Drive — restoring...")
    processed_local.mkdir(parents=True, exist_ok=True)
    if len(list(processed_local.glob('*.wav'))) == 0:
        shutil.copytree(str(processed_drive), str(processed_local))
    guitarset_samples = []
    for wav in sorted(processed_local.glob('*.wav')):
        mid = wav.with_suffix('.mid')
        if mid.exists():
            guitarset_samples.append({
                'audio': str(wav), 'midi': str(mid),
                'source': wav.stem, 'string': 0,
                'duration': 0, 'num_notes': 0
            })
    print(f"   Restored {len(guitarset_samples)} samples — skipping processing.")
    print("   ⏭️  Run Cell G4 next.")

else:
    # ── Full processing ───────────────────────────────────────────────────────

    def create_guitarset_dataset(max_files=360, strings_per_file=6):
        """
        Process GuitarSet files - extract strings and create MIDI labels

        Args:
            max_files: Number of audio files to process (360 = all)
            strings_per_file: Number of strings per file (6 for guitar)
        """

        output_dir = '/content/music_data/guitarset_processed'
        os.makedirs(output_dir, exist_ok=True)

        audio_files_to_process = audio_files[:max_files]

        samples = []
        sample_idx = 0

        print(f"🎸 Processing {len(audio_files_to_process)} GuitarSet files...")
        print(f"   Extracting {strings_per_file} strings per file")
        print(f"   Expected samples: ~{len(audio_files_to_process) * strings_per_file}")
        print(f"   This will take ~10-15 minutes")
        print(f"   Output: {output_dir}\n")

        for audio_file in tqdm(audio_files_to_process, desc="Processing files"):
            # Get corresponding JAMS annotation
            jams_path = gs_parser.get_jams_path(audio_file)

            if jams_path is None:
                continue

            # Parse JAMS to get notes
            notes, duration = gs_parser.parse_jams(jams_path)

            if len(notes) == 0:
                continue

            # Process each of the 6 strings
            for string_idx in range(strings_per_file):
                try:
                    # Extract this string's audio
                    string_audio = gs_parser.extract_string_audio(audio_file, string_idx)

                    # Limit to 30 seconds (saves space & speeds training)
                    max_samples = 30 * 22050
                    if len(string_audio) > max_samples:
                        string_audio = string_audio[:max_samples]
                        duration = min(duration, 30.0)

                    # Save audio file
                    audio_path = f'{output_dir}/sample_{sample_idx:04d}.wav'
                    sf.write(audio_path, string_audio, 22050)

                    # Create MIDI file from JAMS notes
                    midi = pretty_midi.PrettyMIDI()
                    instrument = pretty_midi.Instrument(program=24)  # Acoustic Guitar

                    # Add notes (filter to duration)
                    for note in notes:
                        if note['end'] <= duration:
                            midi_note = pretty_midi.Note(
                                velocity=note['velocity'],
                                pitch=note['pitch'],
                                start=note['start'],
                                end=note['end']
                            )
                            instrument.notes.append(midi_note)

                    midi.instruments.append(instrument)

                    # Save MIDI file
                    midi_path = f'{output_dir}/sample_{sample_idx:04d}.mid'
                    midi.write(midi_path)

                    # Record sample info
                    samples.append({
                        'audio': audio_path,
                        'midi': midi_path,
                        'duration': duration,
                        'num_notes': len(instrument.notes),
                        'source': audio_file.name,
                        'string': string_idx
                    })

                    sample_idx += 1

                except Exception as e:
                    if sample_idx < 5:
                        print(f"   ⚠️ Error on {audio_file.name} string {string_idx}: {e}")
                    continue

        print(f"\n✅ Processing complete!")
        print(f"   Created: {len(samples)} monophonic samples")
        print(f"   Total duration: {sum(s['duration'] for s in samples) / 60:.1f} minutes")
        print(f"   Average notes per sample: {np.mean([s['num_notes'] for s in samples]):.1f}")

        return samples

    # Process ALL GuitarSet files
    print("=" * 70)
    print("🎸 FULL GUITARSET PROCESSING")
    print("=" * 70)
    print()

    guitarset_samples = create_guitarset_dataset(max_files=360, strings_per_file=6)

    # ── Backup to Drive immediately after processing ──────────────────────────
    if len(guitarset_samples) > 0 and not processed_drive.exists():
        print("\n💾 Backing up processed data to Drive...")
        shutil.copytree(str(processed_local), str(processed_drive))
        print(f"✅ Backed up {len(list(processed_drive.glob('*.wav')))} files to Drive.")

    # Show detailed statistics
    if len(guitarset_samples) > 0:
        durations   = [s['duration'] for s in guitarset_samples]
        note_counts = [s['num_notes'] for s in guitarset_samples]

        print(f"\n📊 Dataset Statistics:")
        print(f"   Total samples: {len(guitarset_samples)}")
        print(f"   Total duration: {sum(durations) / 60:.1f} minutes")
        print(f"   Avg duration per sample: {np.mean(durations):.1f}s")
        print(f"   Avg notes per sample: {np.mean(note_counts):.1f}")
        print(f"   Min/Max notes: {min(note_counts)} / {max(note_counts)}")

        print(f"\n🎸 By Guitar String:")
        for string_num in range(6):
            string_samples = [s for s in guitarset_samples if s['string'] == string_num]
            if len(string_samples) > 0:
                string_names = ['E (low)', 'A', 'D', 'G', 'B', 'E (high)']
                print(f"   String {string_num} ({string_names[string_num]}): {len(string_samples)} samples")

        print(f"\n🎵 Sample 0 (first training example):")
        print(f"   Source file: {guitarset_samples[0]['source']}")
        print(f"   String: {guitarset_samples[0]['string']}")
        print(f"   Duration: {guitarset_samples[0]['duration']:.1f}s")
        print(f"   Notes: {guitarset_samples[0]['num_notes']}")
        print(f"   Audio: {guitarset_samples[0]['audio']}")
        print(f"   MIDI: {guitarset_samples[0]['midi']}")

        print(f"\n🔊 Playing sample audio...")
        from IPython.display import Audio
        import librosa
        audio_data, sr = librosa.load(guitarset_samples[0]['audio'])
        display(Audio(audio_data, rate=sr))

        print("\n" + "=" * 70)
        print("✅ DATASET READY FOR TRAINING!")
        print("=" * 70)
        print("\n🚀 Next: Run Cell G4 to split into train/val/test")

    else:
        print("\n❌ No samples created! Check file paths and formats.")

In [ ]:
"""
CELL G4: Split GuitarSet into Train/Val/Test (80/10/10)
"""

from sklearn.model_selection import train_test_split

# Get paths from processed samples
gs_audio = [s['audio'] for s in guitarset_samples]
gs_midi = [s['midi'] for s in guitarset_samples]

print(f"📊 Total GuitarSet samples: {len(gs_audio)}")

# Split: 80% train, 10% val, 10% test
gs_train_audio, gs_temp_audio, gs_train_midi, gs_temp_midi = train_test_split(
    gs_audio, gs_midi, test_size=0.2, random_state=42
)

gs_val_audio, gs_test_audio, gs_val_midi, gs_test_midi = train_test_split(
    gs_temp_audio, gs_temp_midi, test_size=0.5, random_state=42
)

print(f"\n📊 GuitarSet Split:")
print(f"   Train: {len(gs_train_audio)} samples ({len(gs_train_audio)/len(gs_audio)*100:.1f}%)")
print(f"   Val:   {len(gs_val_audio)} samples ({len(gs_val_audio)/len(gs_audio)*100:.1f}%)")
print(f"   Test:  {len(gs_test_audio)} samples ({len(gs_test_audio)/len(gs_audio)*100:.1f}%)")

# Create PyTorch datasets
gs_train_dataset = MusicDataset(gs_train_audio, gs_train_midi, extractor, parser)
gs_val_dataset = MusicDataset(gs_val_audio, gs_val_midi, extractor, parser)
gs_test_dataset = MusicDataset(gs_test_audio, gs_test_midi, extractor, parser)

# Create dataloaders
from torch.utils.data import DataLoader

gs_train_loader = DataLoader(
    gs_train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

gs_val_loader = DataLoader(
    gs_val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

gs_test_loader = DataLoader(
    gs_test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

print("\n✅ GuitarSet dataloaders created!")
print(f"   Train batches: {len(gs_train_loader)}")
print(f"   Val batches: {len(gs_val_loader)}")
print(f"   Test batches: {len(gs_test_loader)}")

# Estimate training time
estimated_time_per_epoch = len(gs_train_loader) * 1.5 + len(gs_val_loader) * 1.0  # seconds
estimated_total = estimated_time_per_epoch * config.NUM_EPOCHS / 60  # minutes

print(f"\n⏱️ Training estimates:")
print(f"   ~{estimated_time_per_epoch:.0f}s per epoch")
print(f"   ~{estimated_total:.0f} minutes for {config.NUM_EPOCHS} epochs")

print("\n🚀 Next: Run Cell G5 to initialize model")

---
## 7. Model Initialisation

Initialises `MusicTranscriptionModel` with random weights for training from scratch on GuitarSet.

The model has **~3.26M parameters** (~13 MB). Three output heads operate in parallel over every frame:
- **Onset head** — binary classifier (note starts here?)
- **Pitch head** — 129-class classifier (which MIDI pitch, or silence?)
- **Velocity head** — regression to normalised note intensity


In [ ]:
"""
CELL G5: Initialize Fresh Model (FROM SCRATCH)
"""

import torch
import os

print("=" * 70)
print("🎸 FRESH MODEL INITIALIZATION (From Scratch)")
print("=" * 70)

# ── Override config for speed ─────────────────────────────────────────────────
config.D_MODEL    = 64
config.N_LAYERS   = 2
config.D_FF       = 128
config.MAX_FRAMES = 256
config.BATCH_SIZE = 32
config.NUM_EPOCHS = 10

# Recreate dataloaders with new batch size
from torch.utils.data import DataLoader

gs_train_loader = DataLoader(
    gs_train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)
gs_val_loader = DataLoader(
    gs_val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)
gs_test_loader = DataLoader(
    gs_test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

# Create model with updated config
gs_model = MusicTranscriptionModel(config).to(device)

total_params     = sum(p.numel() for p in gs_model.parameters())
trainable_params = sum(p.numel() for p in gs_model.parameters() if p.requires_grad)

print(f"\n✅ Model initialized with RANDOM weights!")
print(f"   Architecture: Transformer-based transcription model")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / 1e6:.1f} MB")
print(f"   Device: {device}")

print(f"\n⚙️ Training configuration (optimized for speed):")
print(f"   Learning rate:  {config.LEARNING_RATE}")
print(f"   Optimizer:      AdamW, weight decay {config.WEIGHT_DECAY}")
print(f"   LR Scheduler:   CosineAnnealing ({config.NUM_EPOCHS} epochs)")
print(f"   Batch size:     {config.BATCH_SIZE}")
print(f"   Epochs:         {config.NUM_EPOCHS}")
print(f"   D_MODEL:        {config.D_MODEL}")
print(f"   N_LAYERS:       {config.N_LAYERS}")
print(f"   MAX_FRAMES:     {config.MAX_FRAMES}")

# Optimizer and scheduler
gs_optimizer = torch.optim.AdamW(
    gs_model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)
gs_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    gs_optimizer,
    T_max=config.NUM_EPOCHS
)
gs_criterion = MultiTaskLoss()

estimated_time_per_epoch = len(gs_train_loader) * 1.5 + len(gs_val_loader) * 1.0
estimated_total = estimated_time_per_epoch * config.NUM_EPOCHS / 60

print(f"\n⏱️ Estimated training time: ~{estimated_total:.0f} minutes")
print("\n" + "=" * 70)
print("✅ READY TO TRAIN!")
print("=" * 70)
print("\n🚀 Run Cell G6 to start training")

---
## 8. Training

Trains the model for 30 epochs using **AdamW** with **CosineAnnealing** LR schedule. The best checkpoint (by val loss) is saved to `/content/checkpoints/best_model_guitarset_scratch.pt`.

| Component | Loss function | Weight |
|-----------|--------------|--------|
| Onset | BCEWithLogitsLoss | 1.0 |
| Pitch | CrossEntropyLoss | 10.0 |
| Velocity | MSELoss | 1.0 |

> ⏱️ Expected training time: ~20–25 minutes on a T4 GPU.


In [ ]:
"""
CELL G6: Train on GuitarSet (From Scratch) — optimized, with checkpoint resume
"""

import time, os, shutil
from pathlib import Path

print("🎸 TRAINING ON GUITARSET")
print("=" * 70)

CHECKPOINT_PATH = os.path.join(config.CHECKPOINT_DIR, 'best_model_guitarset_scratch.pt')
RESUME_PATH     = os.path.join(config.CHECKPOINT_DIR, 'resume_checkpoint.pt')

gs_history = {
    'train_loss': [], 'val_loss': [],
    'train_onset': [], 'train_pitch': [], 'train_velocity': [],
    'val_onset': [], 'val_pitch': [], 'val_velocity': []
}
gs_best_val_loss  = float('inf')
start_epoch       = 0
PATIENCE          = 4
epochs_no_improve = 0

# ── Restore checkpoints from Drive ───────────────────────────────────────────
ckpt_drive = Path('/content/drive/MyDrive/guitarset_checkpoints')
ckpt_drive.mkdir(exist_ok=True)
for f in ckpt_drive.glob('*.pt'):
    dest = Path(config.CHECKPOINT_DIR) / f.name
    if not dest.exists():
        shutil.copy2(str(f), str(dest))
        print(f"🔄 Restored {f.name} from Drive")

# ── Resume if checkpoint exists ───────────────────────────────────────────────
if os.path.exists(RESUME_PATH):
    print(f"\n🔄 Resuming from checkpoint...")
    resume = torch.load(RESUME_PATH, map_location=device)
    gs_model.load_state_dict(resume['model_state_dict'])
    gs_optimizer.load_state_dict(resume['optimizer_state_dict'])
    gs_scheduler.load_state_dict(resume['scheduler_state_dict'])
    gs_history        = resume['history']
    gs_best_val_loss  = resume['best_val_loss']
    epochs_no_improve = resume.get('epochs_no_improve', 0)
    start_epoch       = resume['epoch'] + 1
    print(f"   Resuming from epoch {start_epoch + 1}/{config.NUM_EPOCHS}")
    print(f"   Best val loss so far: {gs_best_val_loss:.4f}")
else:
    print("\n🆕 Starting fresh training run")

# ── Training loop ─────────────────────────────────────────────────────────────
start_time = time.time()

for epoch in range(start_epoch, config.NUM_EPOCHS):
    epoch_start = time.time()

    print(f"\nEpoch {epoch + 1}/{config.NUM_EPOCHS}")
    print("-" * 70)

    train_metrics = train_epoch(gs_model, gs_train_loader, gs_criterion, gs_optimizer)
    val_metrics   = validate_epoch(gs_model, gs_val_loader, gs_criterion)
    gs_scheduler.step()

    gs_history['train_loss'].append(train_metrics['total'])
    gs_history['val_loss'].append(val_metrics['total'])
    gs_history['train_onset'].append(train_metrics['onset'])
    gs_history['train_pitch'].append(train_metrics['pitch'])
    gs_history['train_velocity'].append(train_metrics['velocity'])
    gs_history['val_onset'].append(val_metrics['onset'])
    gs_history['val_pitch'].append(val_metrics['pitch'])
    gs_history['val_velocity'].append(val_metrics['velocity'])

    epoch_time = time.time() - epoch_start
    print(f"Train: {train_metrics['total']:.4f} "
          f"(onset: {train_metrics['onset']:.4f}, pitch: {train_metrics['pitch']:.4f}, vel: {train_metrics['velocity']:.4f})")
    print(f"Val:   {val_metrics['total']:.4f} "
          f"(onset: {val_metrics['onset']:.4f}, pitch: {val_metrics['pitch']:.4f}, vel: {val_metrics['velocity']:.4f})")
    print(f"Time:  {epoch_time:.1f}s | ETA: ~{epoch_time * (config.NUM_EPOCHS - epoch - 1) / 60:.0f}min")

    # Save best model
    if val_metrics['total'] < gs_best_val_loss:
        gs_best_val_loss  = val_metrics['total']
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': gs_model.state_dict(),
            'optimizer_state_dict': gs_optimizer.state_dict(),
            'val_loss': val_metrics['total'],
            'history': gs_history,
            'config': config,
        }, CHECKPOINT_PATH)
        print("💾 Saved best model")
    else:
        epochs_no_improve += 1
        print(f"   No improvement ({epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print(f"\n⏹️ Early stopping — no improvement for {PATIENCE} epochs.")
            break

    # Save resume checkpoint every epoch
    torch.save({
        'epoch': epoch,
        'model_state_dict': gs_model.state_dict(),
        'optimizer_state_dict': gs_optimizer.state_dict(),
        'scheduler_state_dict': gs_scheduler.state_dict(),
        'history': gs_history,
        'best_val_loss': gs_best_val_loss,
        'epochs_no_improve': epochs_no_improve,
    }, RESUME_PATH)

    # Backup to Drive every 2 epochs
    if (epoch + 1) % 2 == 0:
        for f in Path(config.CHECKPOINT_DIR).glob('*.pt'):
            shutil.copy2(str(f), str(ckpt_drive / f.name))
        print(f"☁️  Backed up checkpoints to Drive")

# ── Final backup ──────────────────────────────────────────────────────────────
for f in Path(config.CHECKPOINT_DIR).glob('*.pt'):
    shutil.copy2(str(f), str(ckpt_drive / f.name))
print("\n☁️  Final checkpoints backed up to Drive")

# ── Summary ───────────────────────────────────────────────────────────────────
total_time = time.time() - start_time
print("\n" + "=" * 70)
print("✅ TRAINING COMPLETE!")
print(f"   Best val loss: {gs_best_val_loss:.4f}")
print(f"   Total time:    {total_time / 60:.1f} minutes")
print("=" * 70)

final_gap   = gs_history['val_loss'][-1] - gs_history['train_loss'][-1]
improvement = (gs_history['val_loss'][0] - gs_best_val_loss) / gs_history['val_loss'][0] * 100
print(f"\n📊 Training Summary:")
print(f"   Epochs run:         {len(gs_history['val_loss'])}")
print(f"   Starting loss:      {gs_history['val_loss'][0]:.4f}")
print(f"   Final loss:         {gs_history['val_loss'][-1]:.4f}")
print(f"   Best loss:          {gs_best_val_loss:.4f}")
print(f"   Improvement:        {improvement:.1f}%")
print(f"   Generalization gap: {final_gap:.4f}")
print("   ✅ Good generalization!" if final_gap < 1.0 else "   ⚠️ Some overfitting" if final_gap < 2.0 else "   ⚠️ Significant overfitting")
print("\n🚀 Next: Run Cell G7 to visualize results")

---
## 9. Visualise Training Curves

Plots total loss and per-component losses (onset, pitch) across epochs for both train and validation splits. A small train/val gap indicates good generalisation.


In [ ]:
"""
CELL G7: Visualize GuitarSet Training
"""

import matplotlib.pyplot as plt

# Convert history values to plain floats in case any are tensors
def to_float(lst):
    return [v.item() if hasattr(v, 'item') else float(v) for v in lst]

train_loss = to_float(gs_history['train_loss'])
val_loss   = to_float(gs_history['val_loss'])
train_onset = to_float(gs_history['train_onset'])
train_pitch = to_float(gs_history['train_pitch'])
val_onset   = to_float(gs_history['val_onset'])
val_pitch   = to_float(gs_history['val_pitch'])
best_val    = float(gs_best_val_loss.item() if hasattr(gs_best_val_loss, 'item') else gs_best_val_loss)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total loss
axes[0].plot(train_loss, label='Train', linewidth=2.5, marker='o', markersize=4)
axes[0].plot(val_loss,   label='Val',   linewidth=2.5, marker='s', markersize=4)
axes[0].axhline(y=best_val, color='red', linestyle='--', alpha=0.5,
                label=f'Best: {best_val:.3f}')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Total Loss', fontsize=12)
axes[0].set_title('GuitarSet Training Progress', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Component losses
axes[1].plot(train_onset, label='Train Onset', linestyle='--', alpha=0.7)
axes[1].plot(train_pitch, label='Train Pitch', linestyle='--', alpha=0.7)
axes[1].plot(val_onset,   label='Val Onset',   linewidth=2)
axes[1].plot(val_pitch,   label='Val Pitch',   linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Component Losses', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Final Results:")
print(f"   Train Loss: {train_loss[-1]:.4f}")
print(f"   Val Loss:   {val_loss[-1]:.4f}")
print(f"   Best Val:   {best_val:.4f}")
print(f"   Train/Val Gap: {val_loss[-1] - train_loss[-1]:.4f}")

if val_loss[-1] - train_loss[-1] < 1.0:
    print("   ✅ Good generalization!")
elif val_loss[-1] - train_loss[-1] < 2.0:
    print("   ⚠️ Some overfitting")
else:
    print("   ⚠️ Significant overfitting")

---
## 10. Inference & Evaluation

Loads the best saved checkpoint and runs inference on a sample from the test set. Reports:
- **Frame-level pitch accuracy** — percentage of frames where predicted pitch matches ground truth
- **Note-level recall / precision** — comparing extracted notes against ground truth note list
- **Side-by-side visualisation** — onset, pitch, and velocity predictions vs. ground truth over time


In [ ]:
"""
CELL G8: Detailed Inference with Note-by-Note Comparison
"""

import matplotlib.pyplot as plt
import numpy as np

# Helper functions
def extract_notes_from_predictions(onset_pred, pitch_pred, velocity_pred, fps=config.FPS, threshold=0.5):
    """Extract discrete notes from frame-level predictions"""
    notes = []
    onset_frames = np.where(onset_pred > threshold)[0]

    for onset_frame in onset_frames:
        note_pitch = pitch_pred[onset_frame]

        if note_pitch == 128:
            continue

        end_frame = onset_frame + 1
        while end_frame < len(pitch_pred) and pitch_pred[end_frame] == note_pitch:
            end_frame += 1

        start_time = onset_frame / fps
        end_time = end_frame / fps
        note_velocity = np.mean(velocity_pred[onset_frame:end_frame])

        notes.append({
            'pitch': int(note_pitch),
            'start': start_time,
            'end': end_time,
            'duration': end_time - start_time,
            'velocity': note_velocity
        })

    return notes

def midi_to_note_name(midi_pitch):
    """Convert MIDI pitch to note name"""
    if midi_pitch == 128:
        return "Silence"
    notes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    octave = (midi_pitch // 12) - 1
    note = notes[midi_pitch % 12]
    return f"{note}{octave}"

# ── Load best model ───────────────────────────────────────────────────────────
checkpoint = torch.load(
    os.path.join(config.CHECKPOINT_DIR, 'best_model_guitarset_scratch.pt'),
    map_location=device,
    weights_only=False
)
gs_model.load_state_dict(checkpoint['model_state_dict'])
gs_model.eval()

print("✅ Loaded best GuitarSet model")
print(f"   Epoch: {checkpoint['epoch'] + 1}")
print(f"   Val loss: {checkpoint['val_loss']:.4f}")

# ── Load test sample ──────────────────────────────────────────────────────────
test_idx    = 0
test_sample = gs_val_dataset[test_idx]  # returns (features, onset, pitch, velocity)

test_features = test_sample[0].unsqueeze(0).to(device)  # (1, T, N_MELS)
gt_onset      = test_sample[1].numpy()
gt_pitch      = test_sample[2].numpy()
gt_velocity   = test_sample[3].numpy()

# ── Predict ───────────────────────────────────────────────────────────────────
with torch.no_grad():
    predictions = gs_model(test_features)

pred_onset    = torch.sigmoid(predictions['onset'][0]).cpu().numpy()
pred_pitch    = predictions['pitch'][0].argmax(dim=-1).cpu().numpy()
pred_velocity = predictions['velocity'][0].cpu().numpy()

# ── Frame-level accuracy ──────────────────────────────────────────────────────
active_frames = gt_pitch != 128
if np.sum(active_frames) > 0:
    pitch_accuracy = np.sum(pred_pitch[active_frames] == gt_pitch[active_frames]) / np.sum(active_frames)
    print(f"\n📊 Frame-level Pitch Accuracy: {pitch_accuracy * 100:.1f}%")

# ── Extract notes ─────────────────────────────────────────────────────────────
pred_notes = extract_notes_from_predictions(pred_onset, pred_pitch, pred_velocity)
gt_notes   = extract_notes_from_predictions(gt_onset, gt_pitch, gt_velocity, threshold=0.9)

print(f"\n{'='*80}")
print("NOTE-BY-NOTE COMPARISON")
print(f"{'='*80}")
print(f"\n📝 Ground Truth Notes: {len(gt_notes)}")
print(f"🎯 Predicted Notes:    {len(pred_notes)}")
print(f"\n{'='*80}")

# Show first 20 notes
max_notes = min(20, max(len(gt_notes), len(pred_notes)))
print(f"\n{'Ground Truth':<30} {'Predicted':<30} {'Match'}")
print("-" * 80)

for i in range(max_notes):
    gt_str   = "-" * 28
    pred_str = "-" * 28
    match    = ""

    if i < len(gt_notes):
        gt_note = gt_notes[i]
        gt_str  = f"{midi_to_note_name(gt_note['pitch']):<4} @ {gt_note['start']:5.2f}s ({gt_note['duration']:.2f}s)"

    if i < len(pred_notes):
        pred_note = pred_notes[i]
        pred_str  = f"{midi_to_note_name(pred_note['pitch']):<4} @ {pred_note['start']:5.2f}s ({pred_note['duration']:.2f}s)"

    if i < len(gt_notes) and i < len(pred_notes):
        time_diff   = abs(gt_notes[i]['start'] - pred_notes[i]['start'])
        pitch_match = gt_notes[i]['pitch'] == pred_notes[i]['pitch']
        if pitch_match and time_diff < 0.5:
            match = "✅"
        elif pitch_match:
            match = "⚠️ (timing)"
        else:
            match = "❌"

    print(f"{gt_str:<30} {pred_str:<30} {match}")

print("-" * 80)

# ── Note-level metrics ────────────────────────────────────────────────────────
if len(gt_notes) > 0 and len(pred_notes) > 0:
    matches = 0
    for gt_note in gt_notes:
        for pred_note in pred_notes:
            if gt_note['pitch'] == pred_note['pitch'] and abs(gt_note['start'] - pred_note['start']) < 0.5:
                matches += 1
                break

    note_recall    = matches / len(gt_notes) * 100
    note_precision = matches / len(pred_notes) * 100
    print(f"\n📊 Note-Level Metrics:")
    print(f"   Recall:    {note_recall:.1f}%")
    print(f"   Precision: {note_precision:.1f}%")

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
time_axis = np.arange(len(pred_onset)) / config.FPS

# Onset
axes[0].plot(time_axis, gt_onset,   label='Ground Truth', linewidth=2, alpha=0.7)
axes[0].plot(time_axis, pred_onset, label='Predicted',    linewidth=2, alpha=0.7)
axes[0].set_ylabel('Onset', fontsize=12)
axes[0].set_title('Onset Detection (GuitarSet)', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Pitch
gt_pitch_filtered   = np.where(gt_pitch == 128,   np.nan, gt_pitch)
pred_pitch_filtered = np.where(pred_pitch == 128, np.nan, pred_pitch)
axes[1].plot(time_axis, gt_pitch_filtered,   label='Ground Truth', marker='o', markersize=2, linewidth=2)
axes[1].plot(time_axis, pred_pitch_filtered, label='Predicted',    marker='x', markersize=2, linewidth=2)
axes[1].set_ylabel('Pitch (MIDI)', fontsize=12)
axes[1].set_title('Pitch Prediction (GuitarSet)', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Velocity
axes[2].plot(time_axis, gt_velocity,   label='Ground Truth', linewidth=2, alpha=0.7)
axes[2].plot(time_axis, pred_velocity, label='Predicted',    linewidth=2, alpha=0.7)
axes[2].set_ylabel('Velocity', fontsize=12)
axes[2].set_xlabel('Time (seconds)', fontsize=12)
axes[2].set_title('Velocity Prediction (GuitarSet)', fontsize=13, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ GuitarSet inference complete!")